# Non-Uniformly Mixed Pollutants — ECON7720 Lecture 04

Interactive simulations for pollution control **when location matters**.

1. **Transfer coefficients** — how source location affects ambient concentration
2. **Uniform tax vs ambient charges** — why a single tax gets the allocation wrong
3. **Ambient permits and trading ratios** — how the permit market adjusts for location
4. **Hotspots** — what goes wrong with location-blind 1:1 trading

---
*ECON7720 — Ecological & Environmental Economics | The University of Queensland | Dr Juan Soto-Diaz*

In [ ]:
%pip install -q ipywidgets matplotlib numpy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from ipywidgets import interact, FloatSlider, IntSlider, Checkbox

# UQ branding
UQ_PURPLE = "#512478"
UQ_CYAN = "#0099CC"
PINK = "#E6308A"
GREEN = "#2EA836"
ORANGE = "#E87722"
COLORS = [UQ_PURPLE, UQ_CYAN, PINK, GREEN, ORANGE, "#00A4BD", "#8B5CF6", "#DC2626"]

---
## 1. Transfer coefficients and ambient concentration

Place sources on a map. Each source $i$ has a **transfer coefficient** $a_i$ that captures how much one tonne of its emissions raises concentration at the receptor $R$:

$$K_R = \sum_i a_i E_i + B$$

Move the sources around and change their emissions to see how ambient concentration responds. A close source with $a_i = 1.0$ contributes far more per tonne than a distant source with $a_i = 0.2$.

In [ ]:
def plot_transfer(a1=1.0, a2=0.5, a3=0.2, E1=10, E2=10, E3=10, B=5, K_target=20):
    """Visualise transfer coefficients and ambient concentration."""
    a = np.array([a1, a2, a3])
    E = np.array([E1, E2, E3])
    contributions = a * E
    K_R = np.sum(contributions) + B

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle("Transfer Coefficients and Ambient Concentration",
                 fontsize=13, fontweight="bold", color=UQ_PURPLE, y=1.02)

    # -- Left: spatial map --
    ax = axes[0]
    ax.set_title("Source map", fontsize=11, color=UQ_PURPLE)
    ax.set_xlim(-0.5, 10.5)
    ax.set_ylim(-0.5, 6.5)
    ax.set_aspect("equal")
    ax.axis("off")

    # Receptor
    ax.plot(9, 3, "s", color="red", ms=14, zorder=10)
    ax.text(9, 2.2, "Receptor $R$", ha="center", fontsize=9, color="red", fontweight="bold")

    # Sources positioned by transfer coefficient (closer = higher a)
    positions = [(9 - 2.5 / max(ai, 0.05), 3 + 1.5 * (idx - 1)) for idx, ai in enumerate(a)]
    names = ["S1", "S2", "S3"]

    for idx, (ai, ei, (sx, sy)) in enumerate(zip(a, E, positions)):
        c = COLORS[idx]
        # Size proportional to emissions
        ms = max(6, min(ei * 1.2, 20))
        ax.plot(sx, sy, "o", color=c, ms=ms, alpha=0.7, zorder=5)
        ax.text(sx, sy + 0.6, f"{names[idx]}\n$a={ai:.1f}$, $E={ei:.0f}$",
                ha="center", fontsize=8, color=c, fontweight="bold")
        # Arrow to receptor
        ax.annotate("", xy=(8.7, 3), xytext=(sx + 0.3, sy),
                    arrowprops=dict(arrowstyle="->", color=c, lw=1.5 * ai, alpha=0.5))

    # Wind arrow
    ax.annotate("", xy=(9.5, 0.5), xytext=(0.5, 0.5),
                arrowprops=dict(arrowstyle="->", color="gray", lw=2))
    ax.text(5, 0.1, "prevailing wind", ha="center", fontsize=8, color="gray")

    # -- Right: concentration breakdown --
    ax2 = axes[1]
    ax2.set_title("Concentration at receptor $R$", fontsize=11, color=UQ_PURPLE)

    labels = [f"S1 ($a_1 E_1 = {contributions[0]:.1f}$)",
              f"S2 ($a_2 E_2 = {contributions[1]:.1f}$)",
              f"S3 ($a_3 E_3 = {contributions[2]:.1f}$)",
              f"Background $B = {B:.0f}$"]
    values = list(contributions) + [B]
    colors = COLORS[:3] + ["gray"]

    bottom = 0
    for val, lab, col in zip(values, labels, colors):
        ax2.bar(0, val, bottom=bottom, color=col, width=0.5, edgecolor="white",
                lw=1.5, label=lab)
        if val > 0.5:
            ax2.text(0, bottom + val / 2, f"{val:.1f}", ha="center", va="center",
                     fontsize=9, color="white", fontweight="bold")
        bottom += val

    # Target line
    ax2.axhline(K_target, color="red", lw=2, ls="--", label=f"Target $\\bar K = {K_target}$")

    # K_R value
    status = "COMPLIANT" if K_R <= K_target else "EXCEEDS TARGET"
    status_color = GREEN if K_R <= K_target else "red"
    ax2.text(0.4, K_R, f"  $K_R = {K_R:.1f}$ — {status}",
             fontsize=10, color=status_color, fontweight="bold", va="center")

    ax2.set_xlim(-0.5, 1.5)
    ax2.set_ylim(0, max(K_R, K_target) * 1.3)
    ax2.set_ylabel("Concentration ($K_R$)")
    ax2.set_xticks([])
    ax2.legend(fontsize=8, loc="upper right")

    plt.tight_layout()
    plt.show()


interact(
    plot_transfer,
    a1=FloatSlider(value=1.0, min=0.1, max=2.0, step=0.1,
                   description="a₁ (close):", style={"description_width": "initial"}),
    a2=FloatSlider(value=0.5, min=0.1, max=2.0, step=0.1,
                   description="a₂ (mid):", style={"description_width": "initial"}),
    a3=FloatSlider(value=0.2, min=0.1, max=2.0, step=0.1,
                   description="a₃ (far):", style={"description_width": "initial"}),
    E1=FloatSlider(value=10, min=0, max=20, step=1,
                   description="E₁ emissions:", style={"description_width": "initial"}),
    E2=FloatSlider(value=10, min=0, max=20, step=1,
                   description="E₂ emissions:", style={"description_width": "initial"}),
    E3=FloatSlider(value=10, min=0, max=20, step=1,
                   description="E₃ emissions:", style={"description_width": "initial"}),
    B=FloatSlider(value=5, min=0, max=15, step=1,
                  description="Background B:", style={"description_width": "initial"}),
    K_target=FloatSlider(value=20, min=5, max=40, step=1,
                         description="Target K̄:", style={"description_width": "initial"}),
);

### Things to try
1. **Cut S1 emissions by 5**: concentration drops by $a_1 \times 5 = 5$. Now cut S3 by 5: concentration drops only $a_3 \times 5 = 1$. Same tonnes, very different impact.
2. **Raise a₁** (move S1 closer): its contribution jumps — location amplifies the damage.
3. **Set all $a_i = 1$**: location doesn't matter — this is a uniformly mixed pollutant.

---
## 2. Uniform tax vs ambient charges

With a **uniform tax**, every source faces the same price per tonne — the regulator ignores location. With **ambient charges** ($t_i = a_i F$), each source is charged in proportion to its damage at the receptor.

This simulation shows:
- **Left panel**: each source's MAC curve with its charge rate. Circles = ambient charge allocation; crosses = uniform tax allocation.
- **Right panel**: total abatement cost and whether the ambient standard is met.

In [ ]:
def solve_ambient_charge(slopes, a_coeffs, K_target, E0, B):
    """
    Cost-effective allocation under ambient charges.
    Equimarginal: MAC_i / a_i = F for all i.
    MAC_i(q_i) = slope_i * q_i, so q_i = a_i * F / slope_i.
    Constraint: sum(a_i * (E0_i - q_i)) + B = K_target
    => sum(a_i * E0_i) + B - sum(a_i * q_i) = K_target
    => sum(a_i * q_i) = sum(a_i * E0_i) + B - K_target
    => F * sum(a_i^2 / slope_i) = sum(a_i * E0_i) + B - K_target
    """
    K0 = np.sum(a_coeffs * E0) + B  # uncontrolled concentration
    required_reduction = max(K0 - K_target, 0)
    denom = np.sum(a_coeffs**2 / slopes)
    F = required_reduction / denom if denom > 0 else 0
    t_i = a_coeffs * F
    q_i = np.minimum(t_i / slopes, E0)  # abatement per source
    cost_i = 0.5 * slopes * q_i**2
    emissions_i = E0 - q_i
    K_achieved = np.sum(a_coeffs * emissions_i) + B
    return F, t_i, q_i, emissions_i, cost_i, K_achieved


def solve_uniform_tax_for_target(slopes, a_coeffs, K_target, E0, B):
    """
    Find the uniform tax t that achieves the same TOTAL abatement as ambient charges.
    Under uniform tax: q_i = t / slope_i.
    We match total abatement: sum(q_i^uniform) = sum(q_i^ambient).
    """
    # First get the ambient-charge allocation
    F, _, q_ambient, _, _, _ = solve_ambient_charge(slopes, a_coeffs, K_target, E0, B)
    total_abatement = np.sum(q_ambient)

    # Uniform tax achieving same total: t * sum(1/slope_i) = total_abatement
    inv_slopes = 1.0 / slopes
    t_uniform = total_abatement / np.sum(inv_slopes) if np.sum(inv_slopes) > 0 else 0
    q_uniform = np.minimum(t_uniform / slopes, E0)
    cost_uniform = 0.5 * slopes * q_uniform**2
    emissions_uniform = E0 - q_uniform
    K_uniform = np.sum(a_coeffs * emissions_uniform) + B
    return t_uniform, q_uniform, emissions_uniform, cost_uniform, K_uniform


def plot_uniform_vs_ambient(slope1=1.5, slope2=0.5, slope3=1.0,
                            a1=1.0, a2=0.5, a3=0.2,
                            K_target=15, B=3):
    """Compare uniform tax vs ambient charges."""
    n = 3
    slopes = np.array([slope1, slope2, slope3])
    a_coeffs = np.array([a1, a2, a3])
    E0 = np.full(n, 10.0)

    F, t_amb, q_amb, e_amb, cost_amb, K_amb = solve_ambient_charge(
        slopes, a_coeffs, K_target, E0, B)
    t_uni, q_uni, e_uni, cost_uni, K_uni = solve_uniform_tax_for_target(
        slopes, a_coeffs, K_target, E0, B)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
    fig.suptitle("Uniform Tax vs Ambient Charges",
                 fontsize=13, fontweight="bold", color=UQ_PURPLE, y=1.02)

    # -- Left: MAC curves with allocations --
    ax = axes[0]
    ax.set_title("Individual sources", fontsize=11, color=UQ_PURPLE)
    a_max = 12
    a_range = np.linspace(0, a_max, 200)

    names = [f"S1 ($a$={a1:.1f})", f"S2 ($a$={a2:.1f})", f"S3 ($a$={a3:.1f})"]
    for i in range(n):
        c = COLORS[i]
        ax.plot(a_range, slopes[i] * a_range, color=c, lw=1.5, alpha=0.7,
                label=names[i])
        # Ambient charge allocation (circle)
        ax.plot(q_amb[i], t_amb[i], "o", color=c, ms=8, zorder=5)
        # Uniform tax allocation (cross)
        ax.plot(q_uni[i], t_uni, "x", color=c, ms=8, mew=2.5, zorder=5)

    # Ambient charge lines
    for i in range(n):
        if t_amb[i] > 0.1:
            ax.axhline(t_amb[i], color=COLORS[i], ls=":", lw=1, alpha=0.4)

    # Uniform tax line
    ax.axhline(t_uni, color="gray", ls="--", lw=1.5, alpha=0.6,
               label=f"Uniform $t$ = {t_uni:.1f}")

    ax.set_xlabel("Abatement $a_i$")
    ax.set_ylabel("$ / unit")
    ax.set_xlim(0, a_max)
    y_top = max(np.max(t_amb) * 2, t_uni * 2.5, 5)
    ax.set_ylim(0, y_top)
    ax.legend(fontsize=7, loc="upper left")

    # Legend for markers
    circle_patch = plt.Line2D([0], [0], marker="o", color="gray", lw=0, ms=7,
                              label="Ambient charge")
    cross_patch = plt.Line2D([0], [0], marker="x", color="gray", lw=0, ms=7,
                             mew=2, label="Uniform tax")
    ax.legend(handles=ax.get_legend_handles_labels()[1] and ax.legend_.legend_handles
              if ax.legend_ else [], fontsize=7, loc="upper left")
    # Re-do legend properly
    handles, labels = ax.get_legend_handles_labels()
    handles += [circle_patch, cross_patch]
    labels += ["Ambient charge", "Uniform tax"]
    ax.legend(handles, labels, fontsize=7, loc="upper left")

    # -- Right: cost and concentration comparison --
    ax2 = axes[1]
    ax2.set_title("Cost and concentration", fontsize=11, color=UQ_PURPLE)

    tc_amb = np.sum(cost_amb)
    tc_uni = np.sum(cost_uni)

    x_pos = [0, 1]
    bar_labels = ["Uniform\ntax", "Ambient\ncharges"]
    bar_costs = [tc_uni, tc_amb]
    bar_K = [K_uni, K_amb]
    bar_colors = ["gray", UQ_CYAN]

    bars = ax2.bar(x_pos, bar_costs, color=bar_colors, width=0.45,
                   edgecolor="white", lw=1.5)

    for x, bar, cost, K in zip(x_pos, bars, bar_costs, bar_K):
        ax2.text(x, bar.get_height() + 0.3, f"Cost: ${cost:.1f}",
                 ha="center", va="bottom", fontsize=9, fontweight="bold", color=UQ_PURPLE)
        meets = "meets" if K <= K_target + 0.01 else "MISSES"
        k_color = GREEN if K <= K_target + 0.01 else "red"
        ax2.text(x, bar.get_height() * 0.5,
                 f"$K_R$ = {K:.1f}\n({meets} target)",
                 ha="center", va="center", fontsize=8, color="white", fontweight="bold")

    ax2.set_ylabel("Total abatement cost ($)")
    ax2.set_xticks(x_pos)
    ax2.set_xticklabels(bar_labels)
    ax2.set_ylim(0, max(tc_uni, tc_amb) * 1.5 + 1)

    # Saving annotation
    if tc_uni > tc_amb + 0.01:
        note = f"Same total abatement\nUniform tax misses target ($K_R$={K_uni:.1f} vs {K_target})" if K_uni > K_target + 0.01 else "Same total abatement, same cost"
    else:
        note = ""

    fig.text(0.5, -0.02, f"$F$ = {F:.2f}  |  Ambient charges: $t_1$={t_amb[0]:.1f}, "
             f"$t_2$={t_amb[1]:.1f}, $t_3$={t_amb[2]:.1f}  |  "
             f"Uniform tax: $t$={t_uni:.1f}  |  Target $\\bar K$={K_target}",
             ha="center", fontsize=9, color=UQ_PURPLE)

    plt.tight_layout()
    plt.show()


interact(
    plot_uniform_vs_ambient,
    slope1=FloatSlider(value=1.5, min=0.3, max=4.0, step=0.1,
                       description="MAC₁ slope:", style={"description_width": "initial"}),
    slope2=FloatSlider(value=0.5, min=0.3, max=4.0, step=0.1,
                       description="MAC₂ slope:", style={"description_width": "initial"}),
    slope3=FloatSlider(value=1.0, min=0.3, max=4.0, step=0.1,
                       description="MAC₃ slope:", style={"description_width": "initial"}),
    a1=FloatSlider(value=1.0, min=0.1, max=2.0, step=0.1,
                   description="a₁ (close):", style={"description_width": "initial"}),
    a2=FloatSlider(value=0.5, min=0.1, max=2.0, step=0.1,
                   description="a₂ (mid):", style={"description_width": "initial"}),
    a3=FloatSlider(value=0.2, min=0.1, max=2.0, step=0.1,
                   description="a₃ (far):", style={"description_width": "initial"}),
    K_target=FloatSlider(value=15, min=5, max=25, step=1,
                         description="Target K̄:", style={"description_width": "initial"}),
    B=FloatSlider(value=3, min=0, max=10, step=1,
                  description="Background B:", style={"description_width": "initial"}),
);

### Things to try
1. **Set all $a_i = 1$**: uniform and ambient charges are identical — location doesn't matter (uniformly mixed case).
2. **Widen the spread** ($a_1 = 1.5$, $a_3 = 0.1$): the uniform tax badly misses the ambient target while the ambient charges hit it exactly.
3. **Make all MAC slopes equal**: cost differences come purely from location, not technology.
4. **Tighten the target**: both instruments must work harder, but the uniform tax increasingly misallocates.

---
## 3. Ambient permits and trading ratios

Under ambient permits, each permit entitles the holder to raise concentration at $R$ by $\Delta K$. A source with transfer coefficient $a_i$ gets $\Delta E_i = \Delta K / a_i$ tonnes per permit.

When sources trade, one tonne from a **close** source is **not** equivalent to one tonne from a **far** source. The **trading ratio** $a_j / a_i$ ensures concentration stays fixed.

Use this simulation to see how a permit trade redistributes emissions while keeping $K_R$ constant.

In [ ]:
def plot_trading_ratio(a1=1.0, a2=0.5, permits_traded=2.0,
                       slope1=1.5, slope2=0.5):
    """
    Show what happens when Source 2 (far) sells ambient permits to Source 1 (close).
    """
    E0 = 10.0

    # Trading ratio: Source 1 gets (a2/a1) tonnes per tonne Source 2 gives up
    ratio = a2 / a1
    delta_E2 = -permits_traded  # S2 reduces emissions by this
    delta_E1 = permits_traded * ratio  # S1 increases by this (less!)

    # Before trade: equal allocation (say, each emits 6)
    E1_before, E2_before = 6.0, 6.0
    E1_after = E1_before + delta_E1
    E2_after = E2_before + delta_E2

    # Concentration
    K_before = a1 * E1_before + a2 * E2_before
    K_after = a1 * E1_after + a2 * E2_after

    # Abatement costs
    q1_before, q2_before = E0 - E1_before, E0 - E2_before
    q1_after, q2_after = E0 - E1_after, E0 - E2_after
    cost1_before = 0.5 * slope1 * q1_before**2
    cost2_before = 0.5 * slope2 * q2_before**2
    cost1_after = 0.5 * slope1 * max(q1_after, 0)**2
    cost2_after = 0.5 * slope2 * max(q2_after, 0)**2

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle(f"Ambient Permit Trade (trading ratio $a_2/a_1$ = {ratio:.2f})",
                 fontsize=13, fontweight="bold", color=UQ_PURPLE, y=1.02)

    # -- Panel 1: emissions before and after --
    ax1 = axes[0]
    ax1.set_title("Emissions", fontsize=11, color=UQ_PURPLE)
    x = np.arange(2)
    w = 0.3
    bars1 = ax1.bar(x - w/2, [E1_before, E2_before], w, color=[UQ_PURPLE, UQ_CYAN],
                    alpha=0.4, edgecolor="white", label="Before")
    bars2 = ax1.bar(x + w/2, [max(E1_after, 0), max(E2_after, 0)], w,
                    color=[UQ_PURPLE, UQ_CYAN], edgecolor="white", label="After")
    ax1.set_xticks(x)
    ax1.set_xticklabels([f"S1 (close\n$a_1$={a1})", f"S2 (far\n$a_2$={a2})"])
    ax1.set_ylabel("Emissions (tonnes)")
    ax1.set_ylim(0, 12)
    ax1.legend(fontsize=8)

    total_before = E1_before + E2_before
    total_after = max(E1_after, 0) + max(E2_after, 0)
    ax1.text(0.5, 11, f"Total: {total_before:.1f} → {total_after:.1f} tonnes",
             ha="center", fontsize=9, color=UQ_PURPLE, fontweight="bold",
             transform=ax1.get_xaxis_transform())

    # -- Panel 2: concentration --
    ax2 = axes[1]
    ax2.set_title("Concentration at $R$", fontsize=11, color=UQ_PURPLE)

    # Stacked bars
    contrib_before = [a1 * E1_before, a2 * E2_before]
    contrib_after = [a1 * max(E1_after, 0), a2 * max(E2_after, 0)]

    ax2.bar(0, contrib_before[0], 0.45, color=UQ_PURPLE, alpha=0.6, label="S1 contrib")
    ax2.bar(0, contrib_before[1], 0.45, bottom=contrib_before[0], color=UQ_CYAN,
            alpha=0.6, label="S2 contrib")
    ax2.bar(1, contrib_after[0], 0.45, color=UQ_PURPLE)
    ax2.bar(1, contrib_after[1], 0.45, bottom=contrib_after[0], color=UQ_CYAN)

    ax2.set_xticks([0, 1])
    ax2.set_xticklabels(["Before", "After"])
    ax2.set_ylabel("$K_R$ (concentration)")
    ax2.set_ylim(0, max(K_before, K_after) * 1.3 + 1)

    # Show K values
    ax2.text(0, K_before + 0.3, f"$K_R$={K_before:.2f}", ha="center",
             fontsize=9, fontweight="bold", color=UQ_PURPLE)
    ax2.text(1, K_after + 0.3, f"$K_R$={K_after:.2f}", ha="center",
             fontsize=9, fontweight="bold", color=UQ_PURPLE)

    diff = abs(K_after - K_before)
    ax2.text(0.5, max(K_before, K_after) * 1.15,
             f"$\\Delta K_R$ = {K_after - K_before:+.3f}",
             ha="center", fontsize=10, color=GREEN if diff < 0.01 else "red",
             fontweight="bold")
    ax2.legend(fontsize=8)

    # -- Panel 3: abatement costs --
    ax3 = axes[2]
    ax3.set_title("Abatement costs", fontsize=11, color=UQ_PURPLE)

    x = np.arange(2)
    w = 0.3
    ax3.bar(x - w/2, [cost1_before, cost2_before], w,
            color=[UQ_PURPLE, UQ_CYAN], alpha=0.4, edgecolor="white", label="Before")
    ax3.bar(x + w/2, [cost1_after, cost2_after], w,
            color=[UQ_PURPLE, UQ_CYAN], edgecolor="white", label="After")
    ax3.set_xticks(x)
    ax3.set_xticklabels(["S1 (close)", "S2 (far)"])
    ax3.set_ylabel("Abatement cost ($)")
    ax3.legend(fontsize=8)

    tc_before = cost1_before + cost2_before
    tc_after = cost1_after + cost2_after
    saving = tc_before - tc_after
    ax3.text(0.5, max(ax3.get_ylim()[1] * 0.9, max(cost1_before, cost2_before, cost1_after, cost2_after) * 1.2),
             f"Total: ${tc_before:.1f} → ${tc_after:.1f}  (saving ${saving:+.1f})",
             ha="center", fontsize=9, color=UQ_PURPLE, fontweight="bold",
             transform=ax3.get_xaxis_transform())

    plt.tight_layout()
    plt.show()


interact(
    plot_trading_ratio,
    a1=FloatSlider(value=1.0, min=0.2, max=2.0, step=0.1,
                   description="a₁ (close):", style={"description_width": "initial"}),
    a2=FloatSlider(value=0.5, min=0.2, max=2.0, step=0.1,
                   description="a₂ (far):", style={"description_width": "initial"}),
    permits_traded=FloatSlider(value=2.0, min=0, max=5, step=0.5,
                               description="Permits traded:",
                               style={"description_width": "initial"}),
    slope1=FloatSlider(value=1.5, min=0.3, max=4.0, step=0.1,
                       description="MAC₁ slope:", style={"description_width": "initial"}),
    slope2=FloatSlider(value=0.5, min=0.3, max=4.0, step=0.1,
                       description="MAC₂ slope:", style={"description_width": "initial"}),
);

### Things to try
1. **Trade 2 permits**: S2 gives up 2 tonnes, S1 gains only $2 \times (a_2/a_1) = 1$ tonne. Total emissions **fall** but concentration is unchanged.
2. **Set $a_1 = a_2$**: the trading ratio becomes 1:1 — one tonne for one tonne (uniformly mixed case).
3. **Increase $a_1$** (move S1 closer): the ratio drops — S1 gets fewer tonnes per permit, because its emissions hit the receptor harder.
4. **Watch the costs**: if S2 has a flat MAC (cheap abater), the trade saves money — both firms benefit.

---
## 4. Hotspots — what goes wrong with 1:1 trading

If permits are traded **1:1** (ignoring transfer coefficients), pollution can migrate toward the receptor, creating a **hotspot** — even though total emissions meet the cap.

This simulation compares:
- **1:1 trading** (location-blind): permits flow to the cheapest abater, which may be the close source.
- **Ratio trading** (ambient permits): the trading ratio prevents concentration from rising.

In [ ]:
def plot_hotspot(a1=1.0, a2=0.3, slope1=2.0, slope2=0.8, cap=12):
    """
    Compare 1:1 trading (hotspot risk) vs ratio-based trading.
    Two sources, each with E0=10.
    """
    E0 = 10.0
    total_E0 = 2 * E0
    total_abatement = max(total_E0 - cap, 0)

    slopes = np.array([slope1, slope2])
    a_coeffs = np.array([a1, a2])

    # --- 1:1 trading (equimarginal on emissions) ---
    inv_slopes = 1.0 / slopes
    p_11 = total_abatement / np.sum(inv_slopes)
    q_11 = p_11 / slopes
    e_11 = E0 - q_11
    K_11 = np.sum(a_coeffs * e_11)
    cost_11 = 0.5 * slopes * q_11**2

    # --- Ratio trading (equimarginal on concentration) ---
    # MAC_i / a_i = F => q_i = a_i * F / slope_i
    # sum(q_i) = total_abatement => F * sum(a_i / slope_i) = total_abatement
    # Wait, we need sum(a_i * q_i) = K_uncontrolled - K_target.
    # But here we fix total emissions = cap, not K.
    # For ratio trading, we want concentration-neutral trades.
    # Let's find the allocation that minimises cost subject to sum(a_i * e_i) = K_target.
    # Set K_target = K at 1:1 allocation initially, then compare.

    # Actually, let's show: given the same total emissions cap,
    # 1:1 trading allocates by MAC equality,
    # ratio trading allocates by MAC/a equality.

    # For ratio trading with the same cap on total emissions:
    # This doesn't directly apply — ratio trading is about concentration caps.
    # Instead, show the problem: 1:1 trading achieves the emissions cap but
    # may violate a concentration target.

    # Uniform allocation
    q_uniform = total_abatement / 2
    e_uniform = E0 - q_uniform
    K_uniform = np.sum(a_coeffs * e_uniform)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5.5))
    fig.suptitle("Hotspot Risk: 1:1 Trading vs Uniform Standard",
                 fontsize=13, fontweight="bold", color=UQ_PURPLE, y=1.02)

    # -- Panel 1: MAC curves and allocations --
    ax1 = axes[0]
    ax1.set_title("Abatement allocation", fontsize=11, color=UQ_PURPLE)
    a_range = np.linspace(0, E0 * 1.1, 200)

    ax1.plot(a_range, slope1 * a_range, color=UQ_PURPLE, lw=2,
             label=f"$MAC_1$ (close, $a_1$={a1})")
    ax1.plot(a_range, slope2 * a_range, color=UQ_CYAN, lw=2,
             label=f"$MAC_2$ (far, $a_2$={a2})")

    # 1:1 allocation
    ax1.plot(q_11[0], p_11, "o", color=UQ_PURPLE, ms=8, zorder=5)
    ax1.plot(q_11[1], p_11, "o", color=UQ_CYAN, ms=8, zorder=5)
    ax1.axhline(p_11, color="gray", ls="--", lw=1, alpha=0.5,
                label=f"1:1 price = {p_11:.1f}")

    # Uniform allocation
    ax1.axvline(q_uniform, color="gray", ls=":", lw=2, alpha=0.5)
    ax1.text(q_uniform + 0.1, slope1 * q_uniform * 0.5, "uniform",
             fontsize=8, color="gray", rotation=90)

    ax1.set_xlabel("Abatement $a_i$")
    ax1.set_ylabel("$ / unit")
    ax1.set_xlim(0, E0 * 1.1)
    ax1.set_ylim(0, max(p_11 * 2.5, 5))
    ax1.legend(fontsize=7, loc="upper left")

    # -- Panel 2: emissions --
    ax2 = axes[1]
    ax2.set_title("Emissions by source", fontsize=11, color=UQ_PURPLE)

    x = np.arange(2)
    w = 0.25
    ax2.bar(x - w, [e_uniform, e_uniform], w, color=[UQ_PURPLE, UQ_CYAN],
            alpha=0.35, edgecolor="white", label="Uniform")
    ax2.bar(x, e_11, w, color=[UQ_PURPLE, UQ_CYAN],
            edgecolor="white", label="1:1 trade")

    for i, (eu, et) in enumerate(zip([e_uniform, e_uniform], e_11)):
        ax2.text(i - w, eu + 0.2, f"{eu:.1f}", ha="center", fontsize=8, color="gray")
        ax2.text(i, et + 0.2, f"{et:.1f}", ha="center", fontsize=8,
                 color=UQ_PURPLE if i == 0 else UQ_CYAN, fontweight="bold")

    ax2.set_xticks(x)
    ax2.set_xticklabels([f"S1 (close\n$a_1$={a1})", f"S2 (far\n$a_2$={a2})"])
    ax2.set_ylabel("Emissions (tonnes)")
    ax2.set_ylim(0, 12)
    ax2.legend(fontsize=8)

    # -- Panel 3: concentration comparison --
    ax3 = axes[2]
    ax3.set_title("Concentration at receptor $R$", fontsize=11, color=UQ_PURPLE)

    bars = ax3.bar(["Uniform\nstandard", "1:1\ntrade"],
                   [K_uniform, K_11],
                   color=["gray", PINK], width=0.45, edgecolor="white", lw=1.5)

    for bar, val in zip(bars, [K_uniform, K_11]):
        ax3.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2,
                 f"$K_R$ = {val:.1f}", ha="center", va="bottom", fontsize=10,
                 fontweight="bold", color=UQ_PURPLE)

    # Show the direction
    if K_11 > K_uniform + 0.01:
        ax3.annotate(f"HOTSPOT\n+{K_11 - K_uniform:.1f}",
                     xy=(1, K_11), xytext=(1.3, K_11 * 0.85),
                     fontsize=10, color="red", fontweight="bold", ha="center",
                     arrowprops=dict(arrowstyle="->", color="red", lw=1.5))
    elif K_11 < K_uniform - 0.01:
        ax3.annotate(f"Improved\n{K_11 - K_uniform:.1f}",
                     xy=(1, K_11), xytext=(1.3, K_11 * 1.15),
                     fontsize=10, color=GREEN, fontweight="bold", ha="center",
                     arrowprops=dict(arrowstyle="->", color=GREEN, lw=1.5))

    ax3.set_ylabel("$K_R$ (concentration)")
    ax3.set_ylim(0, max(K_uniform, K_11) * 1.4)

    fig.text(0.5, -0.02,
             f"Total emissions cap = {cap}  |  Both allocations meet the cap  |  "
             f"But 1:1 trading shifts emissions toward the receptor",
             ha="center", fontsize=9, color=UQ_PURPLE)

    plt.tight_layout()
    plt.show()


interact(
    plot_hotspot,
    a1=FloatSlider(value=1.0, min=0.2, max=2.0, step=0.1,
                   description="a₁ (close):", style={"description_width": "initial"}),
    a2=FloatSlider(value=0.3, min=0.1, max=2.0, step=0.1,
                   description="a₂ (far):", style={"description_width": "initial"}),
    slope1=FloatSlider(value=2.0, min=0.3, max=5.0, step=0.1,
                       description="MAC₁ slope (close):",
                       style={"description_width": "initial"}),
    slope2=FloatSlider(value=0.8, min=0.3, max=5.0, step=0.1,
                       description="MAC₂ slope (far):",
                       style={"description_width": "initial"}),
    cap=IntSlider(value=12, min=4, max=18, step=1,
                  description="Emissions cap:", style={"description_width": "initial"}),
);

### Things to try
1. **Default settings** ($a_1=1.0$, $a_2=0.3$, steep $MAC_1$): S1 is close but expensive, so 1:1 trading shifts permits to S1 — it abates less, emits more near the receptor → **hotspot**.
2. **Flip the MAC slopes** ($MAC_1$ flat, $MAC_2$ steep): now S1 (close) is the cheap abater, so 1:1 trading moves emissions *away* from the receptor → concentration actually improves.
3. **Set $a_1 = a_2$**: the pollutant is uniformly mixed — 1:1 trading is fine, no hotspot.
4. **Tighten the cap**: the hotspot effect intensifies because more abatement must be reallocated.
5. **Key lesson**: whether 1:1 trading creates a hotspot depends on the *correlation* between transfer coefficients and MAC slopes. It goes wrong when expensive sources are close to the receptor.